In [ ]:
# Setup: five-label intent classification only
import hashlib
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks" / "06camelbert_intent.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from inside the CallSense project.")

SEED = 42
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
INTENT_COLS = ["billing", "technical", "cancellation", "complaint", "information"]
MAX_LENGTH = 128
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 5
PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# Require the Apple Silicon GPU; never select CPU or CUDA as a fallback.
if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS GPU is required. Select the Python (CallSense) kernel in VS Code "
        "and verify torch.backends.mps.is_available() is True."
    )
device = torch.device("mps")
print("Device:", device)


In [ ]:
# Load finalized reviewed labels; CSV inputs are never written
train_path = PROJECT_ROOT / "data/train_reviewed.csv"
original_train_path = PROJECT_ROOT / "data/processed/train.csv"
val_path = PROJECT_ROOT / "data/processed/val.csv"

train_df_reviewed = pd.read_csv(train_path)
original_train_df = pd.read_csv(original_train_path)
val_df = pd.read_csv(val_path)

for name, frame in [("reviewed training", train_df_reviewed),
                    ("original training", original_train_df), ("validation", val_df)]:
    if "complaint_id" not in frame or frame["complaint_id"].isna().any():
        raise ValueError(f"{name}: missing complaint IDs")
    if frame["complaint_id"].duplicated().any():
        raise ValueError(f"{name}: duplicate complaint IDs")
    if frame.empty:
        raise ValueError(f"{name}: empty dataset")

reviewed_ids = set(train_df_reviewed["complaint_id"])
if reviewed_ids != set(original_train_df["complaint_id"]):
    raise ValueError("Reviewed IDs must exactly match original training IDs")
if reviewed_ids & set(val_df["complaint_id"]):
    raise ValueError("Training and validation complaint IDs overlap")

for name, frame in [("training", train_df_reviewed), ("validation", val_df)]:
    required = ["model_text", *INTENT_COLS]
    if not set(required).issubset(frame.columns):
        raise ValueError(f"{name}: missing text or intent columns")
    if frame["model_text"].isna().any() or frame["model_text"].str.strip().eq("").any():
        raise ValueError(f"{name}: missing or empty text")
    if not frame[INTENT_COLS].isin([0, 1]).all().all():
        raise ValueError(f"{name}: intent labels must be binary")

# Record the exact input versions used by this run.
input_hashes = {
    str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in [train_path, original_train_path, val_path]
}
print("Reviewed training:", len(train_df_reviewed), "Validation:", len(val_df))
print(train_df_reviewed[INTENT_COLS].sum())


In [ ]:
# Build fresh tensors from reviewed labels, preserving row alignment
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)

def make_dataset(frame):
    encoded = tokenizer(
        frame["model_text"].tolist(), truncation=True, padding="max_length",
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    labels = torch.tensor(frame[INTENT_COLS].to_numpy(), dtype=torch.float32)
    return TensorDataset(encoded["input_ids"], encoded["attention_mask"], labels)

train_dataset = make_dataset(train_df_reviewed)
val_dataset = make_dataset(val_df)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# Fresh pretrained base model and new optimizer; no fine-tuned checkpoint loaded
torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    num_labels=len(INTENT_COLS),
    problem_type="multi_label_classification",
    id2label=dict(enumerate(INTENT_COLS)),
    label2id={label: i for i, label in enumerate(INTENT_COLS)},
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


In [ ]:
# Train and select the checkpoint with minimum validation loss
history = []
best_val_loss = float("inf")
best_model_state = None
best_epoch = None
wait = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    for input_ids, attention_mask, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids.to(device), attention_mask=attention_mask.to(device),
            labels=labels.to(device),
        )
        loss = outputs.loss
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        train_loss_sum += loss.item() * labels.size(0)

    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:
            outputs = model(
                input_ids=input_ids.to(device), attention_mask=attention_mask.to(device),
                labels=labels.to(device),
            )
            if not torch.isfinite(outputs.loss).item():
                raise RuntimeError("Non-finite validation loss")
            val_loss_sum += outputs.loss.item() * labels.size(0)

    train_loss = train_loss_sum / len(train_dataset)
    val_loss = val_loss_sum / len(val_dataset)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"Epoch {epoch}: train loss={train_loss:.4f}, validation loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        candidate = {name: value.detach().cpu().clone()
                     for name, value in model.state_dict().items()}
        if not all(torch.isfinite(value).all().item()
                   for value in candidate.values() if value.is_floating_point()):
            raise RuntimeError("Checkpoint contains non-finite weights")
        best_model_state = candidate
        best_val_loss = val_loss
        best_epoch = epoch
        wait = 0
    else:
        wait += 1
    if wait >= PATIENCE:
        print("Early stopping")
        break

if best_model_state is None:
    raise RuntimeError("No valid checkpoint was selected")


In [ ]:
# Restore best weights BEFORE generating evaluation probabilities
model.load_state_dict(best_model_state)
model.to(device)
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, labels in val_loader:
        logits = model(
            input_ids=input_ids.to(device), attention_mask=attention_mask.to(device)
        ).logits
        probs = torch.sigmoid(logits)
        if not torch.isfinite(probs).all().item():
            raise RuntimeError("Non-finite validation probabilities")
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
y_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_labels).astype(int)
print("Restored epoch:", best_epoch, "Validation loss:", best_val_loss)


In [ ]:
# Threshold calibration on development data; these are not test scores
threshold_grid = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
best_thresholds = []
for i, intent in enumerate(INTENT_COLS):
    scores = [f1_score(y_true[:, i], (y_probs[:, i] >= t).astype(int), zero_division=0)
              for t in threshold_grid]
    threshold = threshold_grid[int(np.argmax(scores))]
    best_thresholds.append(threshold)
    print(f"{intent}: threshold={threshold:.2f}")

y_pred_tuned = (y_probs >= np.array(best_thresholds)).astype(int)
macro_f1 = f1_score(y_true, y_pred_tuned, average="macro", zero_division=0)
print(f"Development macro F1 (threshold-tuned): {macro_f1:.4f}")
print(classification_report(y_true, y_pred_tuned, target_names=INTENT_COLS, zero_division=0))


In [ ]:
# Read-only error analysis; label corrections belong in the reviewed source before training
errors_df = val_df.copy().reset_index(drop=True)
for i, intent in enumerate(INTENT_COLS):
    errors_df[f"{intent}_actual"] = y_true[:, i]
    errors_df[f"{intent}_predicted"] = y_pred_tuned[:, i]
    errors_df[f"{intent}_probability"] = y_probs[:, i]
display(errors_df.loc[(y_true != y_pred_tuned).any(axis=1)])


In [ ]:
# Export the best inference checkpoint into a new directory; never overwrite a run
model.load_state_dict(best_model_state)
model.eval()
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
save_dir = PROJECT_ROOT / "models" / f"camelbert_intent_reviewed_{run_id}"
save_dir.mkdir(parents=True, exist_ok=False)
model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

settings = {
    "model_name": MODEL_NAME,
    "intent_cols": INTENT_COLS,
    "thresholds": dict(zip(INTENT_COLS, best_thresholds)),
    "threshold_grid": threshold_grid,
    "threshold_comparison": ">=",
    "probability_function": "sigmoid",
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs_requested": EPOCHS,
    "patience": PATIENCE,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "checkpoint_selection": "minimum_validation_loss",
    "evaluation_split": "validation/development; thresholds tuned on this split",
    "development_macro_f1": float(macro_f1),
    "input_sha256": input_hashes,
    "device": str(device),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "history": history,
}
with (save_dir / "settings.json").open("w", encoding="utf-8") as handle:
    json.dump(settings, handle, ensure_ascii=False, indent=2, allow_nan=False)
print("Saved best intent model and settings:", save_dir)
